In [0]:
# MEMBUAT SCHEMA GOLD DAN JUGA MEMBUAT SALES PER MONTH DI GOLD
from pyspark.sql import functions as F
s = lambda t: spark.table(f"{catalog}.silver.{t}")
catalog = "databrics_ecommerce"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

s = lambda t: spark.table(f"{catalog}.silver.{t}")
def save(df, name):
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.gold.{name}")
    print(name, df.count())

orders = s("orders").filter("order_status = 'delivered'")
items = s("order_items")

# 1. sales per month
sales_per_month = (orders.join(items, "order_id")
                   .groupBy(F.month("order_purchase_timestamp").alias("bulan"))
                   .agg(
                        F.round(F.sum("price"), 2).alias("revenue"),
                        F.round(F.sum("freight_value"), 2).alias("ongkir"),
                        F.round(F.sum("price") / F.countDistinct("order_id"), 2).alias("rata2_nilai_order"))
                        .orderBy("bulan"))
save(sales_per_month, "sales_per_month")



In [0]:
# TOP PRODUCT CATEGORY

def save(df, name):
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.gold.{name}")
    print(name, df.count())

orders = s("orders").filter("order_status = 'delivered'")
items = s("order_items")

top_category = (orders.join(items, "order_id")
                .join(s("products"),"product_id")
                .join(s("product_category_name_translation"),"product_category_name")
                .withColumn("kategori", F.coalesce("product_category_name_english","product_category_name", F.lit("unknown")))
                .groupBy("kategori")
                .agg(F.round(F.sum("price"), 2).alias("revenue"), F.count("*").alias("item_terjual"),F.countDistinct("order_id").alias("jumlah_order"))
                .orderBy(F.desc("revenue")))
save(top_category, "top_category")

top_category 71


In [0]:
# SELLER PERFORMANCE

delay = (orders.filter("order_delivered_customer_date IS NOT NULL")
         .select("order_id", (F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"))
         .cast("int"). alias("telat")))

seller_perf = (items.join(delay, "order_id")
               .groupBy("seller_id")
               .agg(F.round(F.sum("price"), 2).alias("revenue"), 
                    F.countDistinct("order_id").alias ("jumlah_order"), 
                    F.round(F.avg("telat") * 100, 2). alias ("persen_telat"))
               .join(s("sellers"), "seller_id", "left")
               .select("seller_id", "seller_city", "seller_state", "revenue", "jumlah_order", "persen_telat")
               .orderBy(F.desc("revenue")))
save(seller_perf, "seller_performance")

seller_performance 2970


In [0]:
# DELIVERY DELAY ANALYSIS 

delivery = (orders.filter("order_delivered_customer_date IS NOT NULL")
            .join(s("customers"), "customer_id")
            .withColumn("hari_kirim", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
            .withColumn("hari_estimasi", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
            .withColumn("selisih_hari", F.datediff("order_delivered_customer_date", "order_estimated_delivery_date"))
            .groupBy(F.to_date(F.date_trunc("month", "order_purchase_timestamp")).alias("bulan"), "customer_state")
            .agg(F.count("*").alias("jumlah_order"),
                 F.round(F.avg("hari_kirim"), 1).alias("rata_rata_hari_kirim"),
                 F.round(F.avg("selisih_hari"), 1).alias("rata2_selisih_hari"),
                 F.round(F.avg((F.col("selisih_hari") > 0).cast("int")) * 100, 2).alias("persen_telat"))
            .orderBy(F.desc("persen_telat")))
save(delivery, "delivery_delay")

delivery_delay 556
